In [1]:
import os

import numpy as np

import mlflow
import mlflow.sklearn
import mlflow.xgboost

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

from imblearn.combine import SMOTETomek
import dagshub

import warnings
warnings.filterwarnings('ignore')




In [2]:
# Load data
iris = load_iris()
X = iris.data
y = iris.target


In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

### Logistic Regression Classifier

In [4]:
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       1.00      0.73      0.85        15
           2       0.79      1.00      0.88        15

    accuracy                           0.91        45
   macro avg       0.93      0.91      0.91        45
weighted avg       0.93      0.91      0.91        45



### Random Forest Classifier

In [5]:
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=3)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.78      0.93      0.85        15
           2       0.92      0.73      0.81        15

    accuracy                           0.89        45
   macro avg       0.90      0.89      0.89        45
weighted avg       0.90      0.89      0.89        45



###  Train XGBoost

In [6]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.88      0.93      0.90        15
           2       0.93      0.87      0.90        15

    accuracy                           0.93        45
   macro avg       0.93      0.93      0.93        45
weighted avg       0.93      0.93      0.93        45



###  Class imbalance using SMOTETomek and then Train XGBoost

In [7]:
smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts=True)

(array([0, 1, 2]), array([35, 35, 35]))

In [8]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.88      0.93      0.90        15
           2       0.93      0.87      0.90        15

    accuracy                           0.93        45
   macro avg       0.93      0.93      0.93        45
weighted avg       0.93      0.93      0.93        45



In [9]:
models = [
    (
        "Logistic Regression", 
        LogisticRegression(C=1, solver='liblinear'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        RandomForestClassifier(n_estimators=30, max_depth=3), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [10]:
reports = []

for model_name, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [11]:

# os.environ['MLFLOW_TRACKING_USERNAME'] = 'sidir13' 
# os.environ['MLFLOW_TRACKING_PASSWORD'] = '!JeanHasaki13!'
# os.environ['MLFLOW_TRACKING_URI'] = 'https://dagshub.com/sidir13/Mlflow_deployement.mlflow'

dagshub.init(repo_owner='sidir13', repo_name='Mlflow_deployement', mlflow=True)



Accessing as sidir13

Initialized MLflow to track repo "sidir13/Mlflow_deployement"

Repository sidir13/Mlflow_deployement initialized!

In [12]:
# Initialize MLflow

mlflow.set_experiment("Target_prediction")
mlflow.set_tracking_uri("https://dagshub.com/sidir13/Mlflow_deployement.mlflow")

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_param("model", model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  

2025/05/01 15:59:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Logistic Regression at: https://dagshub.com/sidir13/Mlflow_deployement.mlflow/#/experiments/0/runs/61adb2a725754e629aabc249ac1283b6
🧪 View experiment at: https://dagshub.com/sidir13/Mlflow_deployement.mlflow/#/experiments/0


2025/05/01 16:00:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Random Forest at: https://dagshub.com/sidir13/Mlflow_deployement.mlflow/#/experiments/0/runs/9f03c92a5ddc44e29d7873cf99577071
🧪 View experiment at: https://dagshub.com/sidir13/Mlflow_deployement.mlflow/#/experiments/0


2025/05/01 16:00:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier at: https://dagshub.com/sidir13/Mlflow_deployement.mlflow/#/experiments/0/runs/16de81f56ac34a3fb6ad95ba88e48d84
🧪 View experiment at: https://dagshub.com/sidir13/Mlflow_deployement.mlflow/#/experiments/0


2025/05/01 16:00:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier With SMOTE at: https://dagshub.com/sidir13/Mlflow_deployement.mlflow/#/experiments/0/runs/09c85123ab1b40cfbf9be34c53f52637
🧪 View experiment at: https://dagshub.com/sidir13/Mlflow_deployement.mlflow/#/experiments/0


### Register the Model

In [13]:
model_name = 'XGB-Smote'
run_id= "fe5ea13f6fa94b8483a3ee46d95be05e"
model_uri = f'runs:/{run_id}/model_name'

with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Successfully registered model 'XGB-Smote'.
2025/05/01 16:01:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


🏃 View run XGBClassifier With SMOTE at: https://dagshub.com/sidir13/Mlflow_deployement.mlflow/#/experiments/0/runs/fe5ea13f6fa94b8483a3ee46d95be05e
🧪 View experiment at: https://dagshub.com/sidir13/Mlflow_deployement.mlflow/#/experiments/0


### Load the model

### Transition the Model to Production

In [14]:
current_model_uri = f"models:/{model_name}/None"
production_model_name = "target-prediction"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

Successfully registered model 'target-prediction'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'target-prediction'.


<ModelVersion: aliases=[], creation_timestamp=1746108088714, current_stage='None', description='', last_updated_timestamp=1746108088714, name='target-prediction', run_id='fe5ea13f6fa94b8483a3ee46d95be05e', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [ ]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}@champion"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
